# Earth climate preprocessing — Death Valley and Wadi Rum

Ported from `notebooks/preprocessing/26_earth_environment_expansion.ipynb`. Original source files are unchanged. Inputs and all generated data stay under `darrel stuff/data/`.

Uses the original 100-subset selection and aliases. The climate processing block is copied from notebook 26; OSM logistics and its dependencies are deliberately not ported.

**Executed locally during verification on 6 November 2026.** Saved outputs come from the local downloaded data, not copied historical results. Read `../../EDA_Notes.md` before rerunning.


In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "outline.md").is_file() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or one of its subfolders")
DATA_ROOT = ROOT / "darrel stuff/data"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "darrel stuff/EDA"))
from environment_workflow import LocalRun as Run, latest_completed_run, select_climate_manifest

print("Source data:", DATA_ROOT)
print("Processed outputs:", DATA_ROOT / "processed")


Source data: c:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\data
Processed outputs: c:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\data\processed


In [2]:
import json
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from scipy.io import netcdf_file
sys.path.insert(0, str(ROOT / "download scripts"))
from download_terraclimate import validate_netcdf
from notebooks.shared.processing import decode_packed, sha256

BASE = DATA_ROOT / "earth/environment_expansion/20261030"
AREAS = {
    "death_valley": {"bbox": (-117.10, 36.0, -116.60, 36.5), "download_id": "region_01"},
    "wadi_rum": {"bbox": (35.2, 29.4, 35.65, 29.8), "download_id": "region_02"},
}
expected = {(spec["download_id"], year, variable, tuple(spec["bbox"]))
            for spec in AREAS.values() for year in range(2015, 2025)
            for variable in ["tmin", "tmax", "ppt", "ws", "srad"]}
CLIMATE_MANIFEST, jobs = select_climate_manifest(BASE / "terraclimate", expected)
aliases = {value["download_id"]: name for name, value in AREAS.items()}
config = {"areas": AREAS, "climate_period": "2015–2024",
          "variables": ["tmin", "tmax", "ppt", "ws", "srad"],
          "climate_weighting": "cos(latitude), native centers inside bbox",
          "scope": "Climate-only port of notebook 26; OSM is excluded"}
print("Accepted manifest:", CLIMATE_MANIFEST)


Accepted manifest: c:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\data\earth\environment_expansion\20261030\terraclimate\manifests\20261108T102210067973Z_run.json


In [3]:
with Run(ROOT, "earth_environment_expansion", config, "Earth/Climate/26_earth_environment_expansion.ipynb") as run:
    for path in [CLIMATE_MANIFEST, ROOT / "download scripts/download_terraclimate.py",
                 ROOT / "notebooks/preprocessing/26_earth_environment_expansion.ipynb"]:
        run.input(path)
    summaries = []
    for job in jobs:
        area = aliases[job['area']]
        bounds = AREAS[area]['bbox']
        assert tuple(job['bounds']) == bounds
        source_path = BASE / 'terraclimate' / job['path']
        receipt_path = source_path.with_suffix('.nc.json')
        receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
        assert receipt['source'] == job['source'] and sha256(source_path) == receipt['sha256']
        metadata = validate_netcdf(source_path, job['variable'], job['year'], bounds)
        run.input(source_path)
        run.input(receipt_path)
        with netcdf_file(str(source_path), 'r', mmap=False, maskandscale=False) as dataset:
            variable = dataset.variables[job['variable']]
            values = decode_packed(variable[:], [getattr(variable, '_FillValue', -32768), getattr(variable, 'missing_value', -32768)],
                                   float(getattr(variable, 'scale_factor', 1)), float(getattr(variable, 'add_offset', 0)))
            lon, lat = np.meshgrid(dataset.variables['lon'][:], dataset.variables['lat'][:])
            west, south, east, north = bounds
            inside = (lon >= west) & (lon < east) & (lat >= south) & (lat < north)
            weights = np.cos(np.deg2rad(lat))
            dates = [datetime(1900, 1, 1) + timedelta(days=float(t)) for t in dataset.variables['time'][:]]
            records = []
            for index, date in enumerate(dates):
                field = values[index]
                valid = inside & np.isfinite(field)
                assert valid.any()
                records.append(pd.DataFrame({'date': date.date().isoformat(), 'longitude': lon[inside], 'latitude': lat[inside],
                                             'value': field[inside], 'valid': valid[inside], 'variable': job['variable'], 'units': metadata['units']}))
                summaries.append({'area': area, 'date': date.date().isoformat(), 'variable': job['variable'], 'units': metadata['units'],
                                  'spatial_weighted_mean': float(np.average(field[valid], weights=weights[valid])),
                                  'spatial_min_of_monthly_field': float(field[valid].min()), 'spatial_max_of_monthly_field': float(field[valid].max()),
                                  'valid_cells': int(valid.sum()), 'selected_cells': int(inside.sum()),
                                  'weighted_coverage': float(weights[valid].sum() / weights[inside].sum())})
            pd.concat(records, ignore_index=True).to_csv(run.path(f"earth/{area}/native_monthly/{job['variable']}_{job['year']}.csv.gz"),
                                                       index=False, compression={'method': 'gzip', 'mtime': 0})
    monthly = pd.DataFrame(summaries)
    assert len(monthly) == 1200 and not monthly.duplicated(['area', 'date', 'variable']).any()
    monthly.to_csv(run.path('regional_monthly_summary.csv'), index=False)
    print(monthly.groupby('area').agg(monthly_fields=('variable', 'size'), minimum_coverage=('weighted_coverage', 'min')))

print("Completed:", run.directory)


              monthly_fields  minimum_coverage
area                                          
death_valley             600               1.0
wadi_rum                 600               1.0
Completed: C:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\data\processed\earth_environment_expansion\20261109T142503798059Z_6a4cf1
